# Avaliação Industrial com IA — Google Colab

Notebook automatizado: clona o repositório, instala dependências, executa análise, treinamento tabular, testes e exporta os artefatos. **Recomendado: T4 GPU** para o treinamento YOLO opcional. A etapa tabular usa CPU.

No Colab, selecione **Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU** antes de executar. A etapa visual é opcional porque baixa um dataset e pode levar bastante tempo.

In [ ]:
# Configuração da execução
RUN_VISION = False  # True para baixar/preparar dataset e treinar YOLO
VISION_EPOCHS = 50
VISION_IMGSZ = 640
VISION_BATCH = 16  # reduza para 8 se ocorrer falta de memória na T4
REPO_URL = 'https://github.com/Augusto-Dantas/avaliacao-industrial-ia.git'
REPO_DIR = '/content/avaliacao-industrial-ia'


In [ ]:
import os, sys, subprocess, pathlib
print('Python:', sys.version)
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
%cd /content/avaliacao-industrial-ia


In [ ]:
# O Colab gerencia o Python do runtime: não criar virtualenv/ensurepip.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-dev.txt'], check=True)
if RUN_VISION:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-vision.txt'], check=True)


In [ ]:
# Análise + Random Forest/Isolation Forest + pytest.
env = os.environ.copy()
env['USE_SYSTEM_PYTHON'] = '1'
env['PYTHON_BIN'] = sys.executable
subprocess.run(['bash', 'scripts/run_pipeline.sh', '--no-install'], env=env, check=True)


In [ ]:
if RUN_VISION:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA indisponível. Selecione T4 GPU no Colab e reinicie o runtime.')
    print('GPU:', torch.cuda.get_device_name(0))
    # Classes genéricas de solda; não comprovam detecção específica de porosidade/HAZ.
    subprocess.run([sys.executable, 'src/setup_yolo.py', '--no-install', '--train', '--datasets', 'rikkarth/welding-defect-object-detection', '--epochs', str(VISION_EPOCHS), '--imgsz', str(VISION_IMGSZ), '--batch', str(VISION_BATCH), '--device', '0', '--save-period', '1'], check=True)
else:
    print('Treinamento YOLO ignorado (RUN_VISION=False).')


In [ ]:
from pathlib import Path
import zipfile
artifact_paths = [Path('reports'), Path('models'), Path('runs/detect/welding-yolo')]
archive = Path('/content/avaliacao-industrial-ia-artifacts.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for base in artifact_paths:
        if base.exists():
            for file in base.rglob('*'):
                if file.is_file() and '.cache' not in file.parts:
                    zf.write(file, file.as_posix())
print(f'Artefatos exportados: {archive} ({archive.stat().st_size / 1024**2:.1f} MiB)')
from google.colab import files
files.download(str(archive))


## Interpretação e limites

- Os modelos tabulares são treinados no CSV industrial versionado.
- O detector visual usa dataset auxiliar com classes genéricas (bad_weld, good_weld, defect). Não alegue que ele identifica especificamente porosidade nem mede HAZ térmica.
- O runtime Colab é temporário. Baixe o ZIP ao final; arquivos em /content podem desaparecer quando a sessão terminar.
